In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import logomaker
import matplotlib.gridspec as gridspec
import os
import scipy.stats
import matplotlib.colors as mcolors

In [2]:
# Used colorpalette
black = '#000000'
lightblack = '#333333'
darkgray = '#666666'
mediumgray = '#999999'
lightgray = '#CCCCCC'

darkred = '#FF0000'
red = '#FF3333'
lightred = '#FF6666'
pink ='#FF9999'
salmon = '#FFCCCC'

In [3]:
def rev_complement(df, extended_alphabet=True):
    if extended_alphabet:
        #rev complement
        rev_colnames = ['T','G','C','A', 'g', 'm']
        df_rev = df.iloc[::-1]
        df_rev.columns = rev_colnames
        df_rev = df_rev.reset_index(drop=True)
        df_rev = df_rev[['A','C','G','T','m','g']]

        return df_rev
    
    else:
        rev_colnames = ['T','G','C','A']
        df_rev = df.iloc[::-1]
        df_rev.columns = rev_colnames
        df_rev = df_rev.reset_index(drop=True)
        df_rev = df_rev[['A','C','G','T']]

        return df_rev
    

In [4]:
def calculate_stats2(dGG):
    """Calculate the letter sizes of mg and CF form dGG logos, But ake only values into account if they are > 0
    (that is if mg or CG are above 0, otherwise it indeicates that both CG and mg are disadvantages for the TF-DNA interaction).
    Calcualte the difference between the pairs, and calculate how large the letter size is in relation to the overall information
    stored at the given position in the logo (expressed in percent). Returns a stats_df comprising average CG, and mg values, 
    the delta between those two, the ratio CG/mg, and the percentages of C, G, m and g.
    """
    CG = []
    mg = []
    ratio_CGmg = []
    delta_s = []
    per_c = []
    per_G = []
    per_m = []
    per_g = []
    
    limit = len(dGG)
    euclidean_norms = dGG.apply(lambda row: row.abs().sum(), axis=1) # euclidean norm over the psam positions
    top_val = dGG.values.max()
    for i in range(limit-1):
        C = dGG.loc[i, 'C']
        G = dGG.loc[i+1, 'G']
        m = dGG.loc[i, 'm']
        g = dGG.loc[i+1, 'g']
        
        delta = (m - C) + (g - G) # calculate delta as suggested by reviewer, if positive, mg favoured, if negative CG.
        delta_s.append(delta)

        if (C > 0) & (G > 0):
            t = (C + G) / 2
            CG.append(t)
        else: 
            CG.append('NA')

        if(m > 0) & (g > 0):
            tm = (m + g) / 2
            mg.append(tm)
        else:
            mg.append('NA')

        try:
            ratio = CG[i] / mg[i]
        except TypeError:
            if (type(CG[i]) == str) & (type(mg[i]) == str):
                ratio = 'not applicable'
            elif type(CG[i]) == str:
                ratio = 'mg available'
            elif type(mg[i]) == str:
                ratio = 'CG available'
        ratio_CGmg.append(ratio)
        
        
        # calculating percentages not based on top value but on the euclidean norm (the absolute spread of the psam)
        try:
            percent_c = C/euclidean_norms[i]
        except TypeError:
            percent_c = 'not applicable'
        per_c.append(percent_c)

        try:
            percent_G = G/euclidean_norms[i+1]
        except TypeError:
            percent_G = 'not applicable'
        per_G.append(percent_G)
        
        try:
            percent_m = m/euclidean_norms[i]
        except TypeError:
            percent_m = 'not applicable'
        per_m.append(percent_m)
        
        try:
            percent_g = g/euclidean_norms[i+1]
        except TypeError:
            percent_g = 'not applicable'
        per_g.append(percent_g)
   

    stats_df = pd.DataFrame({'CG': CG, 
                             'mg': mg, 
                             'ratio CG/mg': ratio_CGmg, 
                             'C%topvalue': per_c,
                             'G%topvalue': per_G,
                             'm%topvalue': per_m,
                             'g%topvalue': per_g,
                             'delta_mg-CG': delta_s
                            })
    return stats_df

In [6]:
base_path = '/home/gralak/updepla/users/gralak/SmileSeq_paper/meSMiLEseq_joint_analysis/'
motif_path = '/04_ProBound_analysis/psam/'
significant_kmer_path = '/02_fishers_exact_test/significant_kmers/'
kmer_ratio_path = '/03_kmer_ratios/'
output_path = '/05_linking_slopes_to_motifs/'

In [ ]:
#import psam
psam = pd.read_csv(base_path + 'SmSAG01/04_ProBound_analysis/psam/POU5F1_FL/binding_mode_size_9/POU5F1_FL_bindingmode_1.csv', index_col=0)

In [ ]:
logo = logomaker.Logo(psam,
                shade_below=0.5,
                fade_below=0.5,
                color_scheme={'A':'#66a61e', 'C':'#7570b3','G':'#ffc809','T':'#d95f02','m':'#a6cee3'},
                    #baseline_width=0
                )

    # style using Logo methods
logo.style_spines(visible=False)
logo.style_spines(spines=['left', 'bottom'], visible=True)
logo.style_xticks(rotation=90, fmt='%d', anchor=0)
    

    # style using Axes methods
logo.ax.set_ylabel("$-\Delta \Delta G$ (kcal/mol)", labelpad=-1)
logo.ax.xaxis.set_ticks_position('none')
logo.ax.xaxis.set_tick_params(pad=-1)
    

    #plt.savefig(os.path.join(out,file.split('.')[0] + '.pdf'))
plt.show()

In [ ]:
# if needed df_rev = rev_complement(df)

In [ ]:
stats_df = calculate_stats2(psam_rev)

In [ ]:
#RATIO! ADAPT and delet
kmer_ratio_df = pd.read_csv('~/updepla/users/gralak/SmileSeq_paper/meSMiLEseq_joint_analysis/SmSAG01/01_kmer_analysis/POU5F1_FL_7mer_enrichment.csv')

In [ ]:
# import p-values and attach them to final_df Import significant kmers
significant = pd.read_csv('~/updepla/users/gralak/SmileSeq_paper/meSMiLEseq_joint_analysis/SmSAG01/02_fishers_exact_test/POU5F1_FL_7kmer_pvalues.csv')

In [ ]:
# keep significant kmers in ratio

final_df['significant'] = final_df['index'].isin(significant_kmers)
# Here I can add a filter to restrict for most significant

N = 50  # Change this to 10, 100, 1000, etc.
most_significant_kmers = final_df.nsmallest(N, ['pval_methl', 'pval_nonmethl'])['index']
final_df['most_significant'] = final_df['index'].isin(most_significant_kmers)

lin_reg = {}
coords = {}
for spec, dataframe in final_df.groupby(['significant', 'CpG']):
    if spec[0]:
        if spec[1]:
            key = 'with_CG'
        else:
            key = 'no_CG'
        
        dataframe_no_nan = dataframe.dropna(subset=['eluted_methl', 'eluted_nonmethl']) #drop nan otherwise no linreg possible
        x_l = dataframe_no_nan.eluted_methl
        y_l = dataframe_no_nan.eluted_nonmethl

        slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(x_l, y_l)
        lin_reg[key] = [slope, intercept, r_value, p_value, std_err]
        x_vals = np.linspace(x_l.min(), x_l.max(), 100)
        y_vals = slope * x_vals + intercept
        coords[key] = {'x': x_vals, 'y': y_vals}

In [ ]:
# PLOT
import matplotlib.colors as mcolors
x = final_df['eluted_methl']
y = final_df['eluted_nonmethl']

fig, ax = plt.subplots(1, 1, figsize=(5.5, 5.5))
#olors = {True : darkred, False : lightblack}
edgecolors = final_df['CpG'].map({True: darkred, False: lightblack})

facecolors = [
    mcolors.to_rgba(edgecolors.iloc[i]) if sig else '#ffffff00'#00 indicates that it is fully transparent
    for i, sig in enumerate(final_df['most_significant'])
]

ax.scatter(x=x, y=y, alpha=0.6, facecolors=facecolors, edgecolors=edgecolors, rasterized = True)
ax.plot(coords['with_CG']['x'], coords['with_CG']['y'], color=darkred)
ax.plot(coords['no_CG']['x'], coords['no_CG']['y'], color=lightblack)

ax.grid(visible=False)

# Remove the top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

lower_limit = None
    # Extent the axis by 5 % of max value
    
if x.nlargest(1).values[0] > y.nlargest(1).values[0]:
    ax.set_xlim(lower_limit,x.nlargest(1).values[0] + (0.05*x.nlargest(1).values[0]))
    ax.set_ylim(lower_limit,x.nlargest(1).values[0] + (0.05*x.nlargest(1).values[0]))

else:
    ax.set_xlim(lower_limit,y.nlargest(1).values[0]+(0.05*y.nlargest(1).values[0]))
    ax.set_ylim(lower_limit,y.nlargest(1).values[0]+(0.05*y.nlargest(1).values[0]))

ax.set_aspect('equal', adjustable='box')
    
    
ax.set_xlabel('methylated DNA', fontfamily='sans-serif', fontsize=10, fontstyle='italic')
ax.set_ylabel('unmethylated DNA', fontfamily='sans-serif', fontsize=10, fontstyle='italic')
    
#ax.set_title(f"{kmer} enrichment, {log} values")
plt.show()

In [ ]:
# save mg or cg letter aaverage letter heights and the slopes. If methyl plus, there should be anticorrelation, if methyl minus, correlation
letter_slope = {}
letter_slope['POU5F1'] = {'mg': max_value_mg.max(), 'slope': lin_reg['CpG'][0]}